# Block 2B: NLP — Damage Report Generation with RAG

**Goal:** Generate professional insurance damage reports using Retrieval-Augmented Generation (RAG).

**Input:** CV damage classification + ML cost estimate

**Approach:** FAISS vector store over NHTSA complaint texts → retrieve relevant context → Claude API generates structured report

**Comparison:** 3 prompt strategies evaluated qualitatively and quantitatively

## Setup

In [ ]:
import os
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import time
import re
from dotenv import load_dotenv

import anthropic
import faiss
from sentence_transformers import SentenceTransformer

load_dotenv('../.env')

DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print('Setup complete.')

## Data Sources

**NHTSA ODI Vehicle Complaints Database** (public API)
- URL: https://api.nhtsa.gov/complaints/complaintsByVehicle
- Contains thousands of real-world vehicle damage and defect descriptions
- Used as the RAG knowledge base: retrieved passages provide domain-specific context
  for report generation

**Simulated case inputs** (from CV + ML block outputs)
- damage_class, confidence, estimated_cost, vehicle metadata

## Build RAG Knowledge Base from NHTSA Data

In [ ]:
def fetch_nhtsa_complaints(makes, years, max_per_query=50):
    """Fetch complaint text from NHTSA ODI API."""
    complaints = []
    base_url = 'https://api.nhtsa.gov/complaints/complaintsByVehicle'

    for make in makes:
        for year in years:
            try:
                resp = requests.get(base_url, params={'make': make, 'modelYear': year}, timeout=10)
                if resp.status_code == 200:
                    data = resp.json().get('results', [])
                    for item in data[:max_per_query]:
                        text = item.get('summary', '') or item.get('description', '')
                        if text and len(text) > 50:
                            complaints.append({
                                'make': make,
                                'year': year,
                                'component': item.get('components', ''),
                                'text': text.strip()
                            })
                time.sleep(0.2)
            except Exception as e:
                print(f'Error fetching {make} {year}: {e}')

    return pd.DataFrame(complaints)

MAKES = ['Toyota', 'Honda', 'Ford', 'BMW', 'Volkswagen']
YEARS = [2018, 2019, 2020, 2021, 2022]

nhtsa_path = DATA_DIR / 'nhtsa_complaints.csv'
if nhtsa_path.exists():
    df_complaints = pd.read_csv(nhtsa_path)
    print(f'Loaded {len(df_complaints)} complaints from cache.')
else:
    print('Fetching NHTSA complaints...')
    df_complaints = fetch_nhtsa_complaints(MAKES, YEARS)
    df_complaints.to_csv(nhtsa_path, index=False)
    print(f'Fetched and saved {len(df_complaints)} complaints.')

df_complaints.head()

In [ ]:
# EDA on text data
df_complaints['text_length'] = df_complaints['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_complaints['text_length'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Complaint Text Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

df_complaints['make'].value_counts().plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Complaints by Car Make')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'nlp_nhtsa_eda.png', dpi=150)
plt.show()

print(f'Total complaints: {len(df_complaints)}')
print(f'Avg text length: {df_complaints["text_length"].mean():.0f} chars')

## Build FAISS Vector Index

In [ ]:
EMBED_MODEL = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBED_MODEL)

texts = df_complaints['text'].tolist()
print(f'Embedding {len(texts)} documents...')
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity via inner product on normalized vecs
faiss.normalize_L2(embeddings)
index.add(embeddings.astype('float32'))

faiss.write_index(index, str(MODELS_DIR / 'nhtsa_faiss.index'))
df_complaints.to_csv(PROCESSED_DIR / 'nhtsa_complaints_processed.csv', index=False)

print(f'FAISS index built: {index.ntotal} vectors, dim={dim}')

In [ ]:
def retrieve(query, k=5):
    q_emb = embedder.encode([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb.astype('float32'), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text': texts[idx],
            'score': float(score),
            'make': df_complaints.iloc[idx]['make']
        })
    return results

# Test retrieval
test_results = retrieve('vehicle door panel dent scratch damage')
print(f'Top retrieval (score={test_results[0]["score"]:.3f}):')
print(test_results[0]['text'][:300])

## Prompt Strategy Comparison

We compare 3 prompting strategies for damage report generation:
- **Strategy A:** Zero-shot (no context, just facts)
- **Strategy B:** RAG-enhanced (with retrieved NHTSA complaint context)
- **Strategy C:** RAG + structured output template (JSON)

In [ ]:
# Sample case from CV + ML block
sample_case = {
    'damage_class': 'dent',
    'confidence': 0.91,
    'estimated_cost_usd': 1250,
    'vehicle_make': 'Toyota',
    'vehicle_model': 'Corolla',
    'vehicle_year': 2020,
    'vehicle_value_usd': 18000,
    'location': 'front left door'
}

def format_case(case):
    return (
        f"Vehicle: {case['vehicle_year']} {case['vehicle_make']} {case['vehicle_model']}\n"
        f"Vehicle value: ${case['vehicle_value_usd']:,}\n"
        f"Damage type: {case['damage_class']} (AI confidence: {case['confidence']:.0%})\n"
        f"Damage location: {case['location']}\n"
        f"Estimated repair cost: ${case['estimated_cost_usd']:,}"
    )

print(format_case(sample_case))

In [ ]:
MODEL_ID = 'claude-haiku-4-5-20251001'

def generate(system_prompt, user_prompt, max_tokens=600):
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': user_prompt}],
        system=system_prompt
    )
    return response.content[0].text

In [ ]:
# Strategy A: Zero-shot
system_A = "You are an insurance claims assessor. Write professional, concise vehicle damage reports."

user_A = f"""Write a professional vehicle damage report based on the following information:

{format_case(sample_case)}

Include: damage description, repair recommendation, and cost assessment."""

report_A = generate(system_A, user_A)
print('=== Strategy A: Zero-shot ===')
print(report_A)

In [ ]:
# Strategy B: RAG-enhanced
query = f"{sample_case['damage_class']} damage {sample_case['vehicle_make']} {sample_case['location']}"
retrieved = retrieve(query, k=3)
context = '\n---\n'.join([r['text'][:400] for r in retrieved])

system_B = """You are an insurance claims assessor with access to historical complaint records.
Use the provided context to write accurate, professional vehicle damage reports."""

user_B = f"""Write a professional vehicle damage report.

VEHICLE INFORMATION:
{format_case(sample_case)}

RELEVANT HISTORICAL COMPLAINTS (for context):
{context}

Write a professional damage report including: damage description, likely cause, repair recommendation, and cost assessment."""

report_B = generate(system_B, user_B)
print('=== Strategy B: RAG-enhanced ===')
print(report_B)

In [ ]:
# Strategy C: RAG + structured JSON output
system_C = """You are an insurance claims assessor. Always respond with valid JSON only.
No explanation outside the JSON."""

user_C = f"""Generate a structured vehicle damage report as JSON.

VEHICLE INFORMATION:
{format_case(sample_case)}

RELEVANT HISTORICAL COMPLAINTS:
{context}

Return this exact JSON structure:
{{
  "report_id": "auto-generated",
  "vehicle": {{"make": "", "model": "", "year": 0, "value_usd": 0}},
  "damage": {{"type": "", "location": "", "severity": "", "description": ""}},
  "assessment": {{"repair_recommendation": "", "estimated_cost_usd": 0, "confidence": ""}},
  "notes": ""
}}"""

report_C_raw = generate(system_C, user_C)
print('=== Strategy C: RAG + Structured JSON ===')
try:
    report_C = json.loads(report_C_raw)
    print(json.dumps(report_C, indent=2))
except json.JSONDecodeError:
    print('(JSON parse error — raw output:)')
    print(report_C_raw)

## Quantitative Evaluation

We evaluate across 10 test cases on:
- **Completeness score:** Does the output contain all required fields? (0–4)
- **Cost accuracy:** Absolute deviation from ML-estimated cost in output
- **Consistency:** Does damage type in report match CV prediction?

In [ ]:
DAMAGE_CLASSES = ['dent', 'scratch', 'crack', 'glass_breakage', 'lamp_breakage', 'tire_flat']
MAKES = ['Toyota', 'Honda', 'Ford', 'BMW', 'Volkswagen']

rng = np.random.RandomState(42)
test_cases = [
    {
        'damage_class': rng.choice(DAMAGE_CLASSES),
        'confidence': round(rng.uniform(0.65, 0.98), 2),
        'estimated_cost_usd': int(rng.uniform(500, 5000)),
        'vehicle_make': rng.choice(MAKES),
        'vehicle_model': 'Sedan',
        'vehicle_year': int(rng.randint(2015, 2023)),
        'vehicle_value_usd': int(rng.uniform(10000, 40000)),
        'location': rng.choice(['front bumper', 'rear door', 'hood', 'side panel', 'windshield'])
    }
    for _ in range(10)
]

REQUIRED_FIELDS = ['damage', 'repair', 'cost', 'vehicle']

def completeness_score(text):
    return sum(1 for f in REQUIRED_FIELDS if f.lower() in text.lower())

def cost_mentioned(text, expected_cost):
    numbers = re.findall(r'\d+', text)
    numbers = [int(n) for n in numbers if len(n) >= 3]
    if not numbers:
        return None
    closest = min(numbers, key=lambda x: abs(x - expected_cost))
    return abs(closest - expected_cost) / expected_cost

results_eval = []
for i, case in enumerate(tqdm(test_cases)):
    case_str = format_case(case)
    q = f"{case['damage_class']} damage {case['vehicle_make']} {case['location']}"
    ctx = '\n---\n'.join([r['text'][:300] for r in retrieve(q, k=2)])

    r_A = generate(system_A, f'Write a professional damage report:\n{case_str}\nInclude damage description, repair recommendation, cost assessment.')
    r_B = generate(system_B, f'Write a professional damage report.\n\nVEHICLE INFORMATION:\n{case_str}\n\nCONTEXT:\n{ctx}\n\nInclude damage description, likely cause, repair recommendation, cost assessment.')

    results_eval.append({
        'case': i,
        'damage_class': case['damage_class'],
        'expected_cost': case['estimated_cost_usd'],
        'completeness_A': completeness_score(r_A),
        'completeness_B': completeness_score(r_B),
        'cost_dev_A': cost_mentioned(r_A, case['estimated_cost_usd']),
        'cost_dev_B': cost_mentioned(r_B, case['estimated_cost_usd']),
    })
    time.sleep(0.5)

df_eval = pd.DataFrame(results_eval)
df_eval.head()

In [ ]:
print('=== Evaluation Results ===')
print(f"Completeness — Strategy A: {df_eval['completeness_A'].mean():.2f}/4")
print(f"Completeness — Strategy B: {df_eval['completeness_B'].mean():.2f}/4")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_eval[['completeness_A', 'completeness_B']].mean().plot(
    kind='bar', ax=axes[0], color=['orange', 'steelblue'])
axes[0].set_title('Average Completeness Score (0–4)')
axes[0].set_xticklabels(['Zero-shot', 'RAG'], rotation=0)
axes[0].set_ylim(0, 4)
axes[0].set_ylabel('Score')

df_eval[['completeness_A', 'completeness_B']].plot(
    kind='box', ax=axes[1], color={'whiskers': 'black', 'caps': 'black', 'medians': 'red', 'boxes': 'steelblue'})
axes[1].set_title('Completeness Score Distribution')
axes[1].set_xticklabels(['Zero-shot', 'RAG'], rotation=0)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'nlp_evaluation.png', dpi=150)
plt.show()

df_eval.to_csv(PROCESSED_DIR / 'nlp_evaluation_results.csv', index=False)

## Error Analysis — Representative Failure Cases

In [ ]:
# Cases where Strategy A underperforms Strategy B
df_eval['completeness_gain'] = df_eval['completeness_B'] - df_eval['completeness_A']
worst_A = df_eval.nsmallest(3, 'completeness_A')
print('Cases where Zero-shot (A) struggled most:')
print(worst_A[['case', 'damage_class', 'completeness_A', 'completeness_B']].to_string(index=False))

## Summary

| Strategy | Completeness | Strengths | Weaknesses |
|---|---|---|---|
| A — Zero-shot | — | Fast, simple | Generic output, misses context |
| B — RAG | — | Domain-grounded, more specific | Slower, depends on retrieval quality |
| C — RAG + JSON | N/A (structured) | Machine-readable, consistent fields | Less narrative quality |

**Selected for deployment:** Strategy C (RAG + JSON) — structured output integrates cleanly into the Streamlit app pipeline.

**Integration:** Receives damage type from CV block and cost estimate from ML block as input. Returns a structured JSON report displayed in the app.